# Example 1 · Bell-Pair Fidelity Landscape

In a Quantum Data Center, the quality of entanglement shared between two QPUs
depends on two physical factors: how noisy the transducer is (κ_T) and how
long the optical fiber is (distance in metres).

This example maps the Bell-pair fidelity across **both dimensions at once**,
producing a 3D surface that shows how every combination of transducer noise
and fiber length affects the entanglement quality.  Two surfaces are shown
side by side:

- **Theoretical** — the analytical prediction from the noise model (Eq. 3 of the paper)
- **Hardware** — measured directly on ibm_torino through the emulation framework

The difference between the two is the additional fidelity loss introduced by
the real quantum processor itself, captured here because the framework runs on
physical qubits.


## Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from QdcEm import Algorithms, RemoteGates, QPU
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from math import sqrt, cos

from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import json, pathlib

# ── Backend ───────────────────────────────────────────────────────────────────
_creds  = json.loads(pathlib.Path("../ibm_credentials.json").read_text())
service = QiskitRuntimeService(channel=_creds["channel"],
                               instance=_creds["instance"],
                               token=_creds["token"])
backend = service.backend("ibm_torino")
_aer    = AerSimulator.from_backend(backend)

# ── QPU layout (physical qubits from ibm_torino coupling map) ─────────────────
#   CommA=3, ENA=14, Proc_A=0  |  CommB=4, ENB=15, Proc_B=7
QPUA = QPU.Make(Comm=3,  EN=14, Processing_Qubits=[0])
QPUB = QPU.Make(Comm=4,  EN=15, Processing_Qubits=[7])
QPUs = [QPUA, QPUB]

shots     = 8_000
simulator = False   # set True to run on AerSimulator

# ── Sweep parameters ──────────────────────────────────────────────────────────
kT_values  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]   # transducer noise levels
dist_steps = list(range(11))                          # 0 … 10 fiber steps (×10 m)
kappa_F    = sqrt(0.01 * 0.0392)                      # G-654-E fiber

print(f"Grid: {len(kT_values)} × {len(dist_steps)} = "
      f"{len(kT_values)*len(dist_steps)} circuits")


## Measure fidelity across the full grid

In [ ]:
my_backend = _aer if simulator else backend
sampler    = SamplerV2(mode=my_backend)

F_hardware   = np.zeros((len(kT_values), len(dist_steps)))
F_analytical = np.zeros((len(kT_values), len(dist_steps)))

for i, kT in enumerate(kT_values):
    for j, n in enumerate(dist_steps):

        # Bell pair prepared through the noisy transducer + fiber channel
        q  = QuantumRegister(6, name='q')
        c  = ClassicalRegister(2, name='c')
        qc = QuantumCircuit(q, c)

        qc.h(q[2])
        qc.cx(q[2], q[0])

        qc.append(Algorithms.M_Unitary(kT), [q[0], q[1]])
        qc.append(Algorithms.M_Unitary(kT), [q[3], q[4]])
        qc.reset(q[1]); qc.reset(q[4])

        qc.append(Algorithms.M_Unitary(kappa_F), [q[0], q[1]])
        qc.append(Algorithms.M_Unitary(kappa_F), [q[3], q[4]])
        for _ in range(n):
            qc.reset(q[1]); qc.reset(q[4])
            qc.append(Algorithms.M_Unitary(kappa_F), [q[0], q[1]])
            qc.append(Algorithms.M_Unitary(kappa_F), [q[3], q[4]])

        qc.cx(q[0], q[3])
        qc.cx(q[3], q[5])
        qc.measure(q[2], c[0])
        qc.measure(q[5], c[1])

        initial_layout = QPU.Get_Initial_Layout(QPUs, QRG=q)
        pm = generate_preset_pass_manager(optimization_level=3,
                                          target=my_backend.target,
                                          initial_layout=initial_layout)
        result = sampler.run([pm.run(qc)], shots=shots)
        print(f"  κ_T={kT:.1f}  steps={n:2d}  job={result.job_id()}")

        data   = result.result()[0].data
        attr   = next(iter(vars(data)))
        counts = getattr(data, attr).get_counts()
        bc     = defaultdict(int)
        for bs, cnt in counts.items():
            bc[''.join(bs[p] for p in (1, 0))] += cnt

        F_hardware[i, j]   = (bc.get('00', 0) + bc.get('11', 0)) / shots
        F_analytical[i, j] = (1 + np.exp(-(n+1) * kappa_F**2)) / 2 * cos(kT)**2

print("\nDone.")


## Plot

In [ ]:
plt.rcParams.update({'font.family': 'serif', 'font.size': 13})

KT, DIST = np.meshgrid(np.array(kT_values),
                       np.array(dist_steps) * 10,
                       indexing='ij')

fig = plt.figure(figsize=(15, 5))

for idx, (data, title, cmap) in enumerate([
    (F_analytical,             "Analytical (Eq. 3)",          'viridis'),
    (F_hardware,               "ibm\_torino Hardware",        'plasma'),
    (F_analytical - F_hardware,"Hardware Overhead (gap)",     'Reds'),
]):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    surf = ax.plot_surface(KT, DIST, data,
                           cmap=cmap, alpha=0.88, edgecolor='none')
    ax.set_xlabel("κ_T",         labelpad=7)
    ax.set_ylabel("Distance (m)", labelpad=7)
    ax.set_zlabel("Fidelity",    labelpad=7)
    ax.set_title(title, fontsize=12)
    ax.set_zlim(0, 1)
    fig.colorbar(surf, ax=ax, shrink=0.45, pad=0.1)

plt.suptitle(
    "Bell-Pair Fidelity Landscape  |  G-654-E Fiber  |  ibm\_torino",
    fontsize=14, y=1.01
)
plt.tight_layout()
plt.savefig("fidelity_landscape.png", dpi=300, bbox_inches='tight')
plt.show()

i05 = kT_values.index(0.5)
print(f"At paper operating point (κ_T=0.5, 10 steps × 10 m):")
print(f"  Analytical : {F_analytical[i05,-1]*100:.1f} %")
print(f"  Hardware   : {F_hardware[i05,-1]*100:.1f} %")
